# TROPOMI and MPAS with UXARRAY directly

First we need to import the driver.

In [1]:
from melodies_monet import driver

In [2]:
# Needed if you want to make changes to `melodies_monet` and don't want to restart kernel:
%load_ext autoreload

%autoreload 2

## Initiate the analysis class

Now lets create an instance of the {mod}`melodies_monet.driver` {class}`~melodies_monet.driver.analysis` class.
It consists of 4 main parts: model instances, observation instances, a paired instance of both.
This will allow us to move things around the plotting function for spatial and overlays and more complex plots.

In [3]:
an = driver.analysis()
an

analysis(
    control='control.yaml',
    control_dict=None,
    models={},
    obs={},
    paired={},
    start_time=None,
    end_time=None,
    time_intervals=None,
    download_maps=True,
    output_dir=None,
    output_dir_save=None,
    output_dir_read=None,
    debug=False,
    save=None,
    read=None,
    regrid=False,
)

## Control File

Read in the required yaml control file that sets up all the definitions of what we want to pair and plot.

In [4]:
an.control = '/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/control_tropomi_l2_CO_mpas_20240101.yaml'
an.read_control()
an.control_dict

{'analysis': {'start_time': '2024-01-01',
  'end_time': '2024-01-02',
  'output_dir': '/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/output/tropomi_mpas',
  'output_dir_save': '/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/output/tropomi_mpas',
  'output_dir_read': '/glade/u/home/lcthompson/mm/MELODIES-MONET/docs/examples/ungridded_support/unstructured_grid_read_uxarray/output/tropomi_mpas',
  'save': {'paired': {'method': 'netcdf',
    'prefix': '20240101',
    'data': 'all'}},
  'read': {'paired': {'method': 'netcdf',
    'filenames': {'tropomi_l2_no2_mpas': ['tropomi_l2_no2_mpas.nc4']}}},
  'debug': False,
  'pairing_kwargs': {'sat_swath_clm': {'apply_ak': True,
    'mod_to_overpass': True}}},
 'model': {'mpas': {'files': ['/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2

## Load the model data 

The driver will automatically loop through the "models" found in the model section of the control file and create model classes for each. Classes include the label, mapping information, and xarray object as well as the filenames.  Note it can open multiple files easily by including wildcards. Here we are only opening one CAM-chem file.

In [5]:
an.open_models()

Mesh file /glade/work/wenfut/MPAS_tools/NCL_codes2/x20.835586.real.asiaaq.init_58L.nc
mpas
['/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L.cam.h2i.2024-01-01-10800.nc', '/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L.cam.h2i.2024-01-01-21600.nc', '/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L.cam.h2i.2024-01-01-32400.nc']
**** Reading CAM unstructured model output (cesm_se / mpas)...
Using unstructured grid file: /glade/work/wenfut/MPAS_tools/NCL_codes2/x20.835586.real.asiaaq.init_58L.nc
Opening unstructured grid with UXArray: /glade/work/wenfut/MPAS_tools/NCL_codes2/x20.835586.real.

/glade/work/lcthompson/conda-envs/melodies-monet/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/glade/work/lcthompson/conda-envs/melodies-monet/lib/python3.11/site-packages/uxarray/core/api.py:505: FutureWarning: In a future version of xarray the default value for data_vars will change from data_vars='all' to data_vars=None. This is likely to lead to different results when multiple datasets have matching variables with overlapping values. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set data_vars explicitly.
  ds = xr.open_mfdataset(paths, chunks=corrected_chunks, **kwargs)


unstructured reader: requested vars not in '/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L.cam.h2i.2024-01-01-10800.nc': ['hyam', 'hybm', 'P0']. Continuing with: ['NO2', 'lat', 'lon', 'lev', 'PS', 'T', 'PDELDRY', 'PMID'].
**** Opened uxarray grid: /glade/work/wenfut/MPAS_tools/NCL_codes2/x20.835586.real.asiaaq.init_58L.nc


In [6]:
an.models

{'mpas': model(
     model='mpas',
     is_global=False,
     radius_of_influence=27794,
     mod_kwargs={'convert_to_ppb': True, 'var_list': ['NO2', 'lat', 'lon', 'lev', 'hyam', 'hybm', 'PS', 'P0', 'T', 'PDELDRY', 'PMID'], 'mesh_file': '/glade/work/wenfut/MPAS_tools/NCL_codes2/x20.835586.real.asiaaq.init_58L.nc'},
     file_str=['/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L.cam.h2i.2024-01-01-10800.nc', '/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L.cam.h2i.2024-01-01-21600.nc', '/glade/campaign/acom/acom-weather/wenfut/MPAS_ASIAAQ/model_output/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L/atm/hist/ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_mg17_58L.cam.h2i.2024-01-01-32400.nc'],
     label='mpas',
     o

In [7]:
an.models['mpas'].obj

<xarray.UxDataset> Size: 2GB
Dimensions:        (z: 58, time: 3, n_face: 835586)
Coordinates:
  * z              (z) float64 464B 35.0 106.2 179.7 ... 3.687e+04 4.112e+04
  * time           (time) datetime64[ns] 24B 2024-01-01T03:00:00 ... 2024-01-...
    latitude       (n_face) float64 7MB dask.array<chunksize=(835586,), meta=np.ndarray>
    longitude      (n_face) float64 7MB dask.array<chunksize=(835586,), meta=np.ndarray>
Dimensions without coordinates: n_face
Data variables:
    NO2            (time, z, n_face) float32 582MB dask.array<chunksize=(1, 58, 835586), meta=np.ndarray>
    pres_pa_mid    (time, z, n_face) float32 582MB dask.array<chunksize=(1, 58, 835586), meta=np.ndarray>
    temperature_k  (time, z, n_face) float32 582MB dask.array<chunksize=(1, 58, 835586), meta=np.ndarray>
    dz_m           (time, z, n_face) float32 582MB dask.array<chunksize=(1, 58, 835586), meta=np.ndarray>
Attributes:
    Conventions:                CF-1.0
    source:                     CAM
    case:                       ASIA-AQ-MPAS_2024_x20.835586.grid_asiaaq_g17_...
    logname:                    wenfut
    host:                       dec0036
    initial_file:               /glade/work/wenfut/MPAS_tools/NCL_codes2/x20....
    topography_file:            /glade/work/wenfut/MPAS_tools/asiaaq_topo/mpa...
    model_doi_url:              not_set
    time_period_freq:           hour_3
    mio_has_unstructured_grid:  True
    mio_mesh_file:              /glade/work/wenfut/MPAS_tools/NCL_codes2/x20....

In [8]:
# All the info in the model class can be called here.
print(an.models['mpas'].label)
print(an.models['mpas'].mapping)

mpas
{'tropomi_l2_no2': {'NO2': 'nitrogendioxide_tropospheric_column'}}


In [9]:
# All the info in the analysis class can also be called.
print(an.start_time)
print(an.end_time)
print(an.download_maps)

2024-01-01 00:00:00
2024-01-02 00:00:00
True


## Open Obs

Now for monet-analysis we will open preprocessed data in either netcdf icartt or some other format.  We will not be retrieving data like monetio does for some observations (ie aeronet, airnow, etc....).  Instead we will provide utitilies to do this so that users can add more data easily.

Like models we list all obs objects in the yaml file and it will loop through and create driver.observation instances that include the model type, file, objects (i.e. data object) and label  

In [10]:
an.control_dict['obs']

{'tropomi_l2_no2': {'debug': True,
  'filename': '/glade/campaign/acom/acom-weather/amirrezaei/tropomi_no2/2024/S5P_OFFL_L2__NO2____20240101T*.nc',
  'sat_type': 'tropomi_l2_no2',
  'obs_type': 'sat_swath_clm',
  'regrid_method': 'nearest_s2d',
  'variables': {'nitrogendioxide_tropospheric_column': {},
   'averaging_kernel': {},
   'air_mass_factor_troposphere': {},
   'air_mass_factor_total': {},
   'tm5_tropopause_pressure': {},
   'latitude_bounds': {},
   'longitude_bounds': {},
   'qa_value': {}}}}

In [11]:
an.open_obs()

Reading TROPOMI L2 NO2 (generic reader)
reading /glade/campaign/acom/acom-weather/amirrezaei/tropomi_no2/2024/S5P_OFFL_L2__NO2____20240101T074458_20240101T092629_32219_03_020600_20240102T234600.nc
reading /glade/campaign/acom/acom-weather/amirrezaei/tropomi_no2/2024/S5P_OFFL_L2__NO2____20240101T092629_20240101T110759_32220_03_020600_20240103T013304.nc
reading /glade/campaign/acom/acom-weather/amirrezaei/tropomi_no2/2024/S5P_OFFL_L2__NO2____20240101T110759_20240101T124929_32221_03_020600_20240103T033227.nc
reading /glade/campaign/acom/acom-weather/amirrezaei/tropomi_no2/2024/S5P_OFFL_L2__NO2____20240101T124929_20240101T143058_32222_03_020600_20240103T050455.nc
reading /glade/campaign/acom/acom-weather/amirrezaei/tropomi_no2/2024/S5P_OFFL_L2__NO2____20240101T143058_20240101T161228_32223_03_020600_20240103T063236.nc
reading /glade/campaign/acom/acom-weather/amirrezaei/tropomi_no2/2024/S5P_OFFL_L2__NO2____20240101T161228_20240101T175358_32224_03_020600_20240103T082604.nc
reading /glade/cam

In [12]:
# All the info in the observation class can also be called.
#an.obs['tempo_l2_no2'].obj
print(an.obs['tropomi_l2_no2'].regrid_method) 

nearest_s2d


## Pair model and obs data

In [13]:
%%time
 
an.pair_data()

1, in pair data


/glade/u/home/lcthompson/mm/MELODIES-MONET/melodies_monet/util/uxarray_util.py:424: RuntimeWarning: Mean of empty slice
  out_arr[..., ti] = np.nanmean(arr[..., sources], axis=-1)
/glade/u/home/lcthompson/mm/MELODIES-MONET/melodies_monet/util/uxarray_util.py:424: RuntimeWarning: Mean of empty slice
  out_arr[..., ti] = np.nanmean(arr[..., sources], axis=-1)
/glade/u/home/lcthompson/mm/MELODIES-MONET/melodies_monet/util/uxarray_util.py:424: RuntimeWarning: Mean of empty slice
  out_arr[..., ti] = np.nanmean(arr[..., sources], axis=-1)
/glade/u/home/lcthompson/mm/MELODIES-MONET/melodies_monet/util/uxarray_util.py:424: RuntimeWarning: Mean of empty slice
  out_arr[..., ti] = np.nanmean(arr[..., sources], axis=-1)
/glade/u/home/lcthompson/mm/MELODIES-MONET/melodies_monet/util/uxarray_util.py:424: RuntimeWarning: Mean of empty slice
  out_arr[..., ti] = np.nanmean(arr[..., sources], axis=-1)
/glade/u/home/lcthompson/mm/MELODIES-MONET/melodies_monet/util/uxarray_util.py:424: RuntimeWarning: 

CPU times: user 6min 22s, sys: 36.2 s, total: 6min 58s
Wall time: 7min 40s


In [14]:
an.paired

{'tropomi_l2_no2_mpas': pair(
     type='sat_swath_clm',
     radius_of_influence=1000000.0,
     obs='tropomi_l2_no2',
     model='mpas',
     model_vars=['NO2'],
     obs_vars=['nitrogendioxide_tropospheric_column'],
     filename='tropomi_l2_no2_mpas.nc',
 )}

In [15]:
an.paired['tropomi_l2_no2_mpas'].obj
# ds = an.paired['tropomi_l2_no2_mpas'].obj
# ds.to_netcdf("tropomi_l2_no2_mpas")

<xarray.Dataset> Size: 147MB
Dimensions:                              (time: 10, n_face: 835586)
Coordinates:
  * time                                 (time) datetime64[ns] 80B 2024-01-01...
    longitude                            (n_face) float64 7MB 119.8 ... 121.0
    latitude                             (n_face) float64 7MB 25.42 ... 20.01
Dimensions without coordinates: n_face
Data variables:
    NO2                                  (time, n_face) float64 67MB nan ... nan
    nitrogendioxide_tropospheric_column  (time, n_face) float64 67MB nan ... nan

In [16]:
df = an.paired['tropomi_l2_no2_mpas'].obj.to_dataframe()

In [17]:
po = an.paired['tropomi_l2_no2_mpas'].obj
for v in ['NO2', 'nitrogendioxide_tropospheric_column']:
    print(v, 'finite per time slice:', po[v].notnull().sum(dim='n_face').values)

m = an.models['mpas'].obj
print('model time range:', m['time'].values.min(), '->', m['time'].values.max())
print('model time dtype:', m['time'].dtype)   # datetime64 vs object(cftime) matters for interp

import numpy as np
from melodies_monet.util.sat_l2_swath_utility import (
    _mod2tropomi_swath, interp_vertical_mod2tropomi)

mod = an.models['mpas'].obj
print("raw model NO2 finite:", int(np.isfinite(mod['NO2'].values).sum()), "/", mod['NO2'].size)
print("model time dtype:", mod['time'].dtype,
      "| range:", mod['time'].values.min(), "->", mod['time'].values.max())
print("pres_pa_mid in model:", 'pres_pa_mid' in mod.variables)

obj  = an.obs['tropomi_l2_no2'].obj
gran = list(obj.values())[0]
gran = gran[0] if isinstance(gran, list) else gran
if 'time' in gran.dims:
    gran = gran.squeeze('time', drop=False)
gt = gran['time'].values
tsel = gt if np.ndim(gt) == 0 else np.asarray(gt).ravel()[0]
print("granule time:", tsel)

mod_t = mod.interp(time=tsel)
print("A) NO2 finite after time interp:", int(np.isfinite(mod_t['NO2'].values).sum()))

gf = mod.attrs.get('mio_scrip_file') or mod.attrs.get('mio_mesh_file')
on_swath = _mod2tropomi_swath(mod_t, gran, 'nearest_s2d', ['NO2', 'pres_pa_mid'], gf)
print("B) NO2 on swath finite:", int(np.isfinite(on_swath['NO2'].values).sum()))
print("   pres_pa_mid on swath finite:", int(np.isfinite(on_swath['pres_pa_mid'].values).sum()))

no2_t = interp_vertical_mod2tropomi(gran, on_swath, ['NO2'])
print("C) NO2 on TROPOMI layers finite:", int(np.isfinite(no2_t['NO2'].values).sum()))

NO2 finite per time slice: [14584 13887 10679 10162 12934 13282  9908 10752 10183  7838]
nitrogendioxide_tropospheric_column finite per time slice: [16775 16533 16961 16843 16535 16778 16733 16628 16550 16570]
model time range: 2024-01-01T03:00:00.000000000 -> 2024-01-01T09:00:00.000000000
model time dtype: datetime64[ns]
raw model NO2 finite: 145391964 / 145391964
model time dtype: datetime64[ns] | range: 2024-01-01T03:00:00.000000000 -> 2024-01-01T09:00:00.000000000
pres_pa_mid in model: True
granule time: 2024-01-01T00:00:00.000000000
A) NO2 finite after time interp: 0
B) NO2 on swath finite: 0
   pres_pa_mid on swath finite: 0
C) NO2 on TROPOMI layers finite: 0


In [21]:
pair = an.paired['tropomi_l2_no2_mpas']  # or your pair key

## Generate plots

In [23]:
%%time

an.plotting()

-5291568243188695.0 5291568243188695.0
-7470662197920298.0 7470662197920298.0
{'color': 'k', 'linestyle': '-', 'marker': '*', 'linewidth': 2.0, 'markersize': 10.0, 'label': 'tropomi_l2_no2', 'fontsize': 14.4}
CPU times: user 2min 20s, sys: 1.2 s, total: 2min 21s
Wall time: 2min 36s


ValueError: 'y' not found in array dimensions ('time', 'n_face')

**10 Figures**

::::{card-carousel} 10

:::{card} Figure 1
:img-background: output/camchem/plot_grp1.timeseries.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.png
:width: 50%
:::

:::{card} Figure 2
:img-background: output/camchem/plot_grp1.timeseries.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.png
:width: 50%
:::

:::{card} Figure 3
:img-background: output/camchem/plot_grp2.taylor.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.png
:width: 50%
:::

:::{card} Figure 4
:img-background: output/camchem/plot_grp2.taylor.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.png
:width: 50%
:::

:::{card} Figure 5
:img-background: output/camchem/plot_grp3.spatial_bias.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 6
:img-background: output/camchem/plot_grp3.spatial_bias.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 7
:img-background: output/camchem/plot_grp4.spatial_overlay.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 8
:img-background: output/camchem/plot_grp4.spatial_overlay.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.airnow_cam-chem.png
:width: 50%
:::

:::{card} Figure 9
:img-background: output/camchem/plot_grp5.boxplot.OZONE.2019-09-01_00.2019-09-09_00.all.CONUS.png
:width: 50%
:::

:::{card} Figure 10
:img-background: output/camchem/plot_grp5.boxplot.OZONE.2019-09-01_00.2019-09-09_00.epa_region.R1.png
:width: 50%
:::

::::